In [3]:
import requests
import pandas as pd
import numpy as np
import ta
from datetime import datetime, timedelta
import time

In [41]:
def parse_moex_data(tickers=None, days=180, interval=24):
    """
    Парсинг данных с Московской биржи с поддержкой разных интервалов
    
    Parameters:
    -----------
    tickers : list, optional
        Список тикеров для парсинга. По умолчанию основные голубые фишки
    days : int, optional
        Количество дней исторических данных (по умолчанию 180)
    interval : int, optional
        Интервал данных в минутах. Доступные значения:
        - 1: 1 минута
        - 10: 10 минут  
        - 60: 1 час
        - 24: 1 день (по умолчанию)
        - 7: 1 неделя
        - 31: 1 месяц
    
    Returns:
    --------
    dict
        Словарь с DataFrame для каждого тикера
    """
    
    # Валидные интервалы MOEX API
    valid_intervals = {
        1: "1 минута",
        10: "10 минут", 
        60: "1 час",
        24: "1 день",
        7: "1 неделя",
        31: "1 месяц"
    }
    
    if interval not in valid_intervals:
        raise ValueError(f"Неверный интервал. Допустимые значения: {list(valid_intervals.keys())}")
    
    if not tickers:
        tickers = ['SBER', 'GAZP', 'LKOH', 'ROSN', 'YDEX', 'VTBR', 'TATN', 'GMKN']
    
    all_data = {}
    
    # Рассчитываем даты
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    
    print(f"=== ПАРСИНГ ДАННЫХ ЗА ПОСЛЕДНИЕ {days} ДНЕЙ ===")
    print(f"📊 Интервал: {valid_intervals[interval]}")
    
    for i, ticker in enumerate(tickers):
        print(f"({i+1}/{len(tickers)}) Получаем данные для {ticker}...")
        
        url = f"https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/{ticker}/candles.json"
        
        params = {
            'from': start_date.strftime('%Y-%m-%d'),
            'till': end_date.strftime('%Y-%m-%d'),
            'interval': interval,
        }
        
        try:
            response = requests.get(url, params=params, timeout=15)
            data = response.json()
            
            if 'candles' in data and data['candles']['data']:
                candles = data['candles']['data']
                
                # Создаем DataFrame
                df = pd.DataFrame(candles, columns=[
                    'open', 'close', 'high', 'low', 'value', 'volume', 'begin', 'end'
                ])
                
                # Обрабатываем даты
                df['date'] = pd.to_datetime(df['begin'])
                df['ticker'] = ticker
                
                # Выбираем нужные колонки и сортируем
                df = df[['date', 'ticker', 'open', 'high', 'low', 'close', 'volume']]
                df = df.sort_values('date').reset_index(drop=True)
                
                all_data[ticker] = df
                print(f"  ✅ {ticker}: {len(df)} записей")
                
            else:
                print(f"  ❌ {ticker}: нет данных в указанный период")
                
        except Exception as e:
            print(f"  ❌ {ticker}: ошибка - {e}")
        
        # Небольшая пауза чтобы не нагружать API
        time.sleep(0.5)
    
    print(f"\n📊 Парсинг завершен. Получено данных для {len(all_data)} тикеров")
    return all_data

stock_data = parse_moex_data(interval=10)
stock_data['GAZP']

=== ПАРСИНГ ДАННЫХ ЗА ПОСЛЕДНИЕ 180 ДНЕЙ ===
📊 Интервал: 10 минут
(1/8) Получаем данные для SBER...
  ❌ SBER: ошибка - HTTPSConnectionPool(host='iss.moex.com', port=443): Max retries exceeded with url: /iss/engines/stock/markets/shares/boards/TQBR/securities/SBER/candles.json?from=2025-04-15&till=2025-10-12&interval=10 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7ff7f589bb00>: Failed to resolve 'iss.moex.com' ([Errno -3] Temporary failure in name resolution)"))
(2/8) Получаем данные для GAZP...
  ❌ GAZP: ошибка - HTTPSConnectionPool(host='iss.moex.com', port=443): Max retries exceeded with url: /iss/engines/stock/markets/shares/boards/TQBR/securities/GAZP/candles.json?from=2025-04-15&till=2025-10-12&interval=10 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7ff7e5f0cd70>: Failed to resolve 'iss.moex.com' ([Errno -3] Temporary failure in name resolution)"))
(3/8) Получаем данные для LKOH...


KeyboardInterrupt: 

In [43]:
from aiomoex import get_board_candles
import asyncio
import aiohttp

VALID_INTERVALS = {
        1: "1 минута",
        10: "10 минут", 
        60: "1 час",
        24: "1 день",
        7: "1 неделя",
        31: "1 месяц"
    }

async def fetch_ticker_data(session: aiohttp.ClientSession, ticker: str, interval: int, start_date: str, end_date: str) -> dict[str, list[dict[str, str | int | float]]]:
    try:
        res = await get_board_candles(session, ticker, interval, start_date, end_date)
        return {ticker: res}
    except Exception as e:
        print(f'Ошибка парсинга. Не удалось получить данные для {ticker}, {e}')
        return {ticker: []}

async def get_moex_data(tickers: list[str], days: int = 180, interval: int = 24):
    if interval not in VALID_INTERVALS:
        raise ValueError(f"Неверный интервал. Допустимые значения: {list(VALID_INTERVALS.keys())}")
    
    end_date = datetime.now()
    start_date = (end_date - timedelta(days=days)).strftime('%Y-%m-%d')
    end_date = end_date.strftime('%Y-%m-%d')
    

    async with aiohttp.ClientSession() as session:
        coros = [fetch_ticker_data(session, ticker, interval, start_date, end_date) for ticker in tickers]
        stock_data = await asyncio.gather(*coros)
    stock_data = {k: v for d in stock_data for k, v in d.items()}
    return stock_data


tickers = ['SBER', 'GAZP', 'LKOH', 'ROSN', 'YDEX', 'VTBR', 'TATN', 'GMKN']

stock_data = await get_moex_data(tickers, interval=31)

sber_df = pd.DataFrame(stock_data['SBER'])
sber_df['ticker'] = 'SBER'
print(sber_df)

Ошибка парсинга. Не удалось получить данные для VTBR, Cannot connect to host iss.moex.com:443 ssl:default [Temporary failure in name resolution]
Ошибка парсинга. Не удалось получить данные для LKOH, Cannot connect to host iss.moex.com:443 ssl:default [Temporary failure in name resolution]
Ошибка парсинга. Не удалось получить данные для GMKN, Cannot connect to host iss.moex.com:443 ssl:default [Temporary failure in name resolution]
Ошибка парсинга. Не удалось получить данные для YDEX, Cannot connect to host iss.moex.com:443 ssl:default [Temporary failure in name resolution]
Ошибка парсинга. Не удалось получить данные для GAZP, Cannot connect to host iss.moex.com:443 ssl:default [Temporary failure in name resolution]
Ошибка парсинга. Не удалось получить данные для TATN, Cannot connect to host iss.moex.com:443 ssl:default [Temporary failure in name resolution]
Ошибка парсинга. Не удалось получить данные для ROSN, Cannot connect to host iss.moex.com:443 ssl:default [Temporary failure in na